# Tableaux croisés : pivot_table (cours)

Exemples à **exécuter** (petit tableau de ventes). Les exercices (ventes magasins) sont tout en bas.

### Un petit df de ventes

In [1]:
import pandas as pd
ventes = pd.DataFrame({
    "region":    ["Nord","Nord","Sud","Sud","Nord"],
    "categorie": ["Informatique","Bureautique","Informatique","Mobilier","Informatique"],
    "montant":   [850, 120, 610, 450, 300],
})
print(ventes)

  region     categorie  montant
0   Nord  Informatique      850
1   Nord   Bureautique      120
2    Sud  Informatique      610
3    Sud      Mobilier      450
4   Nord  Informatique      300


### pivot_table : région (lignes) x catégorie (colonnes)

In [12]:
grille = ventes.pivot_table(index="region", columns="categorie", values="montant", aggfunc="sum")
print(grille)   # cases sans données = NaN

categorie  Bureautique  Informatique  Mobilier
region                                        
Nord             120.0        1150.0       NaN
Sud                NaN         610.0     450.0


### Même calcul qu'un groupby, mais en grille

In [3]:
print(ventes.groupby(["region","categorie"])["montant"].sum())   # même chose, en liste

region  categorie   
Nord    Bureautique      120
        Informatique    1150
Sud     Informatique     610
        Mobilier         450
Name: montant, dtype: int64


### Cases vides -> 0, et totaux

In [4]:
print(ventes.pivot_table(index="region", columns="categorie", values="montant",
                         aggfunc="sum", fill_value=0, margins=True, margins_name="Total"))

categorie  Bureautique  Informatique  Mobilier  Total
region                                               
Nord               120          1150         0   1270
Sud                  0           610       450   1060
Total              120          1760       450   2330


# Exercices

Données : `data/ventes_magasins.csv` (ventes avec `region`, `categorie`, `trimestre`, `montant`). **À toi de choisir tes outils.**

1. Grille **région (lignes) × catégorie (colonnes)** = **CA total**. Une case est vide : repère-la, explique pourquoi, et remplis-la avec une valeur qui a du sens ici.
2. Sur cette grille, ajoute les **totaux** par région et par catégorie. Quelle région fait le plus gros CA ? Quelle catégorie ?
3. Refais la grille région × catégorie avec le **montant moyen** d'une vente (pas le total). Qu'est-ce qui change par rapport à la question 1 ?
4. Grille **région (lignes) × trimestre (colonnes)** du CA total. Un trimestre ressort-il (saisonnalité) ?
5. Réflexion (pas de code) : différence entre `groupby` et `pivot_table`, et quand préférer l'un plutôt que l'autre ?

In [5]:
# --- CELLULE FOURNIE : données chargées pour toi ---
import pandas as pd
ventes_mag = pd.read_csv("data/ventes_magasins.csv")
print(ventes_mag.shape)
ventes_mag.head()

(67, 4)


,region,categorie,trimestre,montant
0,Nord,Accessoires,T1,127.92
1,Nord,Mobilier,T2,286.93
2,Nord,Bureautique,T4,54.53
3,Nord,Informatique,T2,462.94
4,Sud,Mobilier,T4,381.05


In [18]:
## 1
df_pivot_1 = ventes_mag.pivot_table(index="region", columns="categorie", values="montant", aggfunc="sum", fill_value=0)
df_pivot_1

# La valeur manquante est la ligne pour la catégorie Mobilier dans la région Ouest
# Etant donné qu'on est avec des valeurs à remplir et qu'on a aucune information sur le montant on va remplir les avec l'argument : fill_value=0

categorie,Accessoires,Bureautique,Informatique,Mobilier
region,,,,
Est,379.56,594.43,2605.14,527.09
Nord,685.14,275.46,2370.33,1515.43
Ouest,787.71,451.38,4030.32,0.00
Sud,357.52,384.75,1379.79,2123.01


In [19]:
## 2
df_pivot_2 = ventes_mag.pivot_table(index="region", columns="categorie", values="montant", aggfunc="sum", fill_value=0, margins=True, margins_name="Total montant")
df_pivot_2

# Meilleur CA (région) : Ouest --> 5269.41
# Meilleur CA (catégorie) : Informatique --> 10385.58

categorie,Accessoires,Bureautique,Informatique,Mobilier,Total montant
region,,,,,
Est,379.56,594.43,2605.14,527.09,4106.22
Nord,685.14,275.46,2370.33,1515.43,4846.36
Ouest,787.71,451.38,4030.32,0.00,5269.41
Sud,357.52,384.75,1379.79,2123.01,4245.07
Total montant,2209.93,1706.02,10385.58,4165.53,18467.06


In [26]:
## 3
df_pivot_3 = ventes_mag.pivot_table(index="region", columns="categorie", values="montant", aggfunc="mean")
df_pivot_3

# Ce qui change c'est les  régions qui ressortent par catégories et leurs écarts.
# A la question 1 les régions avec le plus gros montant sont respectivement : Ouest, Est, Ouest, Sud avec des écarts allant jusqu'à 2 voir 3 fois le montant en fonction des régions
# A la question 2 les régions avec la moyenne des montants les plus élevés sont respectivement : Est, Sud, Sud, Nord avec de tout petis écarts entre les régions

categorie,Accessoires,Bureautique,Informatique,Mobilier
region,,,,
Est,126.520000,84.918571,651.2850,263.5450
Nord,114.190000,91.820000,592.5825,378.8575
Ouest,112.530000,75.230000,671.7200,NaN
Sud,119.173333,96.187500,689.8950,353.8350


In [ ]:
## 4
df_pivot_4 = ventes_mag.pivot_table(index="region", columns="trimestre", values="montant", aggfunc="sum", fill_value=0)
df_pivot_4

# Je vois pas spécialement de trismestre qui attire l'oeil comparé aux autres mis à part le T3 où on peut voir des chutes drastiques des montants générés dans les régions Est et Sud

trimestre,T1,T2,T3,T4
region,,,,
Est,1122.93,1108.92,160.80,1713.57
Nord,1509.40,1336.87,1247.88,752.21
Ouest,589.08,399.35,1589.00,2691.98
Sud,638.51,1493.86,84.69,2028.01


In [ ]:
## 5 (réflexion en commentaire)
# Group by : regroupe les données d'une table autour d'une série de 1 ou plusieurs colonnes et aggrège sur une autre colonne 
# pivot_table : même chose pour l'aggrégation mais entre 2 colonnes et les tranbsforme en tableau croisées
# Ex : group by du montant total par rapport au région et trimestre veut dire que pour chaque couple saison + trimestre on aura une ligne avec la somme dzes CA pour chaque couple alors que la même chose avec un
# pivot_table va faire l'aggrégation mais va renvoyer les donnzer sous forme de tableau croisé donc une ligne par région et une colonne par catégorie (dans les exercices) 